In [ ]:
# Cell 1: Database Connection
import os
from sqlalchemy import create_engine, text

# --- CONFIGURATION ---
# TODO: Update these with your PostgreSQL credentials
DB_USER = "postgres"
DB_PASS = "your_password_here"  # Change this to your PostgreSQL password
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "postgres"

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

try:
    engine = create_engine(DATABASE_URL)
    # Test the connection
    with engine.connect() as conn:
        res = conn.execute(text("SELECT 1")).scalar()
        print(f"✅ Database Connection Successful! (Test result: {res})")
except Exception as e:
    print(f"❌ Connection Failed: {e}")

✅ Database Connection Successful! (Test result: 1)


In [ ]:
import time
import datetime
import pandas as pd
import numpy as np
import pyotp
import requests
import json
from SmartApi import SmartConnect
from tqdm.notebook import tqdm # For the progress bar
import logging

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')
logger = logging.getLogger()

# ==========================================
# 1. MULTI-KEY CONFIGURATION
# ==========================================
# Add your 5 API Keys here. 
# NOTE: The CLIENT_CODE and PASSWORD usually remain the same, 
# but if you have 5 different accounts, change those too.
# TODO: Add your Angel One API credentials here
# Get your API keys from: https://smartapi.angelbroking.com/
# You can add up to 5 API keys for load balancing and rate limit handling
CREDENTIALS_POOL = [
    {"api_key": "your_api_key_1_here", "client": "your_client_code_here", "pwd": "your_password_here", "totp": "your_totp_secret_here"},
    {"api_key": "your_api_key_2_here", "client": "your_client_code_here", "pwd": "your_password_here", "totp": "your_totp_secret_here"},
    {"api_key": "your_api_key_3_here", "client": "your_client_code_here", "pwd": "your_password_here", "totp": "your_totp_secret_here"},
    {"api_key": "your_api_key_4_here", "client": "your_client_code_here", "pwd": "your_password_here", "totp": "your_totp_secret_here"},
    {"api_key": "your_api_key_5_here", "client": "your_client_code_here", "pwd": "your_password_here", "totp": "your_totp_secret_here"},
]

class AngelKeyManager:
    def __init__(self, creds_list):
        self.sessions = []
        self.current_index = 0
        print(f"Initializing {len(creds_list)} API Sessions...")
        
        for cred in creds_list:
            try:
                obj = SmartConnect(api_key=cred['api_key'])
                totp = pyotp.TOTP(cred['totp']).now()
                data = obj.generateSession(cred['client'], cred['pwd'], totp)
                if data['status']:
                    self.sessions.append(obj)
                    print(f" >> Session Active for Key: ...{cred['api_key'][-4:]}")
                else:
                    print(f" >> Failed for Key: ...{cred['api_key'][-4:]} | {data['message']}")
            except Exception as e:
                print(f" >> Error: {e}")
        
        if not self.sessions:
            raise Exception("No API Sessions could be established!")

    def get_session(self):
        """Returns a session and rotates the index"""
        session = self.sessions[self.current_index]
        # Rotate to next key
        self.current_index = (self.current_index + 1) % len(self.sessions)
        return session

# Initialize the Pool (Run this once)
key_manager = AngelKeyManager(CREDENTIALS_POOL)
print("Setup Complete. Uncomment the initialization line above after filling credentials.")

Initializing 5 API Sessions...


[I 251125 02:42:31 smartConnect:124] in pool


 >> Session Active for Key: ...PxjD


[I 251125 02:42:32 smartConnect:124] in pool


 >> Session Active for Key: ...LJaB


[I 251125 02:42:34 smartConnect:124] in pool


 >> Session Active for Key: ...nLR0


[I 251125 02:42:36 smartConnect:124] in pool


 >> Session Active for Key: ...p7sc


[I 251125 02:42:37 smartConnect:124] in pool


 >> Session Active for Key: ...lboe
Setup Complete. Uncomment the initialization line above after filling credentials.


In [7]:
# CELL 3: Fetch NSE Official CSV & Initialize Master Skeleton (Strict Raw Mode)
import requests
import pandas as pd
import json
import os
import io
import numpy as np
from datetime import datetime

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Official NSE Equity List
NSE_CSV_URL = "https://nsearchives.nseindia.com/content/equities/EQUITY_L.csv"

DATA_DIR = 'data'
OUTPUT_FILE = os.path.join(DATA_DIR, 'NSE_STOCKS_SKELETON.json')

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': '*/*',
    'Connection': 'keep-alive'
}

# ==========================================
# 2. FETCH AND PARSE CSV (STRICT)
# ==========================================
def fetch_nse_csv():
    print(f"[{datetime.now()}] 📥 Downloading NSE Equity CSV...")
    try:
        response = requests.get(NSE_CSV_URL, headers=HEADERS, timeout=15)
        
        if response.status_code == 200:
            raw_text = response.content.decode('utf-8')
            
            # Debug: Count exact lines from server
            total_lines = len(raw_text.splitlines())
            print(f"🔹 Raw content received from Server: {total_lines} lines")
            
            # Read CSV content
            # keep_default_na=False prevents empty strings becoming NaN (we handle it manually)
            csv_content = io.StringIO(raw_text)
            df = pd.read_csv(csv_content, keep_default_na=False)
            
            # Clean column names (strip spaces)
            df.columns = [c.strip() for c in df.columns]
            
            print(f"✅ Pandas Parsed Rows: {len(df)}")
            
            # Mismatch Check
            # (Total Lines - 1 Header) should equal df rows. 
            # If Raw is 2227, DataFrame should be 2226.
            if len(df) != (total_lines - 1):
                print(f"⚠️ Warning: Potential Row Mismatch! Raw: {total_lines-1} vs Parsed: {len(df)}")
                print("   (This might be due to empty trailing lines in the CSV file)")
            
            return df
        else:
            print(f"❌ Failed to download. Status Code: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"❌ Exception during download: {e}")
        return None

# ==========================================
# 3. BUILD MASTER JSON OBJECTS
# ==========================================
def build_master_skeleton(df):
    master_list = []
    
    print(f"[{datetime.now()}] 🏗️ Building Master JSON Structure...")
    
    for _, row in df.iterrows():
        # Clean basic fields
        symbol = str(row.get('SYMBOL', '')).strip()
        series = str(row.get('SERIES', '')).strip()
        name = str(row.get('NAME OF COMPANY', '')).strip()
        isin = str(row.get('ISIN NUMBER', '')).strip()
        listing_date = str(row.get('DATE OF LISTING', '')).strip()
        
        # Handle Numeric conversions safely
        try:
            mkt_lot = int(row.get('MARKET LOT', 1))
        except:
            mkt_lot = 1
            
        # --- THE MASTER OBJECT ---
        stock_obj = {
            "master_key": symbol,
            
            # 1. Normalized Fields
            "identifiers": {
                "symbol": symbol,
                "isin": isin,
                "slb_isin": None,
                "company_name": name,
                "active_series": [series] 
            },
            
            "trading_info": {
                "token_nse": None,
                "token_bse": None,
                "tick_size": None,
                "lot_size": mkt_lot, 
                "listing_date": listing_date,
                "currency": "INR",
                "segment": "EQUITY"
            },
            
            "classification": {
                "industry": None,
                "sector": None,
                "market_cap_category": None,
                "market_cap_value": None
            },
            
            "flags": {
                "is_fno": False,
                "is_etf": False,
                "is_slb_eligible": False,
                "is_mtf_eligible": False,
                "is_suspended": False,
                "is_delisted": False,
                "is_debt": False,
                "is_municipal_bond": False
            },
            
            # 2. RAW DATA STORAGE (Store Everything)
            "__RAW_DATA_SOURCES__": {
                "source_nse_csv": {
                    "SYMBOL": symbol,
                    "NAME OF COMPANY": name,
                    "SERIES": series,
                    "DATE OF LISTING": listing_date,
                    "PAID UP VALUE": row.get('PAID UP VALUE'),
                    "MARKET LOT": row.get('MARKET LOT'),
                    "ISIN NUMBER": isin,
                    "FACE VALUE": row.get('FACE VALUE')
                },
                "source_angel_api": None,
                "source_nse_web": None 
            }
        }
        
        master_list.append(stock_obj)
        
    return master_list

# ==========================================
# 4. EXECUTION
# ==========================================
def main():
    df = fetch_nse_csv()
    
    if df is not None:
        data = build_master_skeleton(df)
        
        # Save to JSON
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4)
            
        print(f"\n" + "="*50)
        print(f"🎉 SKELETON CREATED: {OUTPUT_FILE}")
        print(f"🔹 Total Items in File : {len(data)}")
        print("="*50)

if __name__ == "__main__":
    main()

[2025-11-25 03:17:06.713797] 📥 Downloading NSE Equity CSV...
🔹 Raw content received from Server: 2203 lines
✅ Pandas Parsed Rows: 2202
[2025-11-25 03:17:07.766558] 🏗️ Building Master JSON Structure...

🎉 SKELETON CREATED: data\NSE_STOCKS_SKELETON.json
🔹 Total Items in File : 2202


In [ ]:
# CELL 4: NSE Indices List Fetcher (JSON + Postgres + Schema Reset)
# DEPENDENCY: Requires Cell 1 (Database Connection) to be run first.

import requests
import json
import os
import time
import random
from datetime import datetime
from sqlalchemy import MetaData, Table, Column, Integer, String, Float, Boolean, text
from sqlalchemy.dialects.postgresql import insert  # Correct lowercase import

# ==========================================
# 1. CONFIGURATION & TOGGLES
# ==========================================
STORE_IN_JSON = True       
STORE_IN_POSTGRES = True   

# File Config
DATA_DIR = 'data'
LOG_DIR = 'logs'
OUTPUT_FILE = os.path.join(DATA_DIR, 'NSE_Indices_List.json')
LOG_FILE = os.path.join(LOG_DIR, 'nse_indices_fetch.log')

if not os.path.exists(DATA_DIR): os.makedirs(DATA_DIR)
if not os.path.exists(LOG_DIR): os.makedirs(LOG_DIR)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': '*/*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate', 
    'Referer': 'https://www.nseindia.com/market-data/live-market-indices',
    'Connection': 'keep-alive',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-origin',
    'X-Requested-With': 'XMLHttpRequest'
}

URL_HOME = "https://www.nseindia.com"
URL_API = "https://www.nseindia.com/api/allIndices"

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def log_message(msg):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(f"[{timestamp}] {msg}\n")
    print(msg)

def save_to_postgres(data_list):
    """Saves indices to Postgres using the existing global 'engine'"""
    if 'engine' not in globals():
        log_message("❌ Error: Database 'engine' not found. Please run Cell 1 first.")
        return False

    try:
        # --- FIX: DROP TABLE TO ENSURE FRESH SCHEMA ---
        # This deletes the old table so we can recreate it with the new columns (like 'key')
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS indices CASCADE"))
        
        metadata = MetaData()
        
        # Define Table Schema
        indices_table = Table(
            'indices', metadata,
            Column('id', Integer, primary_key=True, autoincrement=True),
            Column('name', String, unique=True, nullable=False),
            Column('key', String), # This caused the error before
            Column('index_symbol', String),
            Column('is_fno_eligible', Boolean),
            Column('last_price', Float),
            Column('percent_change', Float),
            Column('previous_close', Float),
            Column('open_price', Float),
            Column('high_price', Float),
            Column('low_price', Float),
            Column('year_high', Float),
            Column('year_low', Float),
            Column('pe_ratio', String),
            Column('pb_ratio', String),
            Column('div_yield', String),
            Column('raw_meta', String)
        )
        
        # Create table newly
        metadata.create_all(engine)
        
        log_message("🗄️  Database Schema Reset. Starting Bulk Upsert...")
        
        with engine.begin() as conn:
            for item in data_list:
                # Prepare record
                record = {
                    "name": item.get("index", "").strip(),
                    "key": item.get("key", ""),
                    "index_symbol": item.get("indexSymbol", ""),
                    "is_fno_eligible": item.get("isDerivativesEligible", False),
                    "last_price": item.get("last"),
                    "percent_change": item.get("percentChange"),
                    "previous_close": item.get("previousClose"),
                    "open_price": item.get("open"),
                    "high_price": item.get("high"),
                    "low_price": item.get("low"),
                    "year_high": item.get("yearHigh"),
                    "year_low": item.get("yearLow"),
                    "pe_ratio": str(item.get("Price-to-Earnings-ratio", "")),
                    "pb_ratio": str(item.get("Price-to-Book-ratio", "")),
                    "div_yield": str(item.get("Dividend-Yield", "")),
                    "raw_meta": json.dumps(item)
                }
                
                # Upsert Logic
                stmt = insert(indices_table).values(record)
                update_stmt = stmt.on_conflict_do_update(
                    index_elements=['name'],
                    set_={
                        "key": stmt.excluded.key,
                        "last_price": stmt.excluded.last_price,
                        "percent_change": stmt.excluded.percent_change,
                        "raw_meta": stmt.excluded.raw_meta,
                        "is_fno_eligible": stmt.excluded.is_fno_eligible
                    }
                )
                conn.execute(update_stmt)
                
        log_message(f"✅ Database Update Complete: {len(data_list)} records.")
        return True
        
    except Exception as e:
        log_message(f"❌ Database Error: {str(e)}")
        return False

# ==========================================
# 3. MAIN LOGIC
# ==========================================
def fetch_indices_manual():
    log_message("🚀 Starting Fetch via Manual Session...")
    session = requests.Session()
    session.headers.update(HEADERS)
    try:
        response_home = session.get(URL_HOME, timeout=15)
        if response_home.status_code != 200: return None
        time.sleep(random.uniform(2, 4))
        response_api = session.get(URL_API, timeout=15)
        if response_api.status_code == 200:
            raw_data = response_api.json()
            if isinstance(raw_data, dict) and "data" in raw_data: data_list = raw_data["data"]
            elif isinstance(raw_data, list): data_list = raw_data
            else: return None
            
            processed_list = []
            rename_map = {"pe": "Price-to-Earnings-ratio", "pb": "Price-to-Book-ratio", "dy": "Dividend-Yield"}
            keys_to_remove = ["chartTodayPath", "chart30dPath", "chart365dPath"]

            for item in data_list:
                if not isinstance(item, dict): continue 
                category_key = item.get("key", "").upper()
                item["isDerivativesEligible"] = "INDICES ELIGIBLE IN DERIVATIVES" in category_key
                for k in keys_to_remove: item.pop(k, None)
                for old_key, new_key in rename_map.items():
                    if old_key in item: item[new_key] = item.pop(old_key)
                processed_list.append(item)
            return processed_list
        return None
    except Exception as e:
        log_message(f"❌ Exception: {str(e)}")
        return None

def main():
    data = fetch_indices_manual()
    if data:
        if STORE_IN_JSON:
            with open(OUTPUT_FILE, 'w', encoding='utf-8') as f: json.dump(data, f, indent=4)
            log_message(f"✅ Saved JSON: {OUTPUT_FILE}")
        if STORE_IN_POSTGRES:
            save_to_postgres(data)
    else:
        log_message("❌ FAILURE: Could not fetch data.")

if __name__ == "__main__":
    main()

🚀 Starting Fetch via Manual Session...
✅ Saved JSON: data\NSE_Indices_List.json
🗄️  Database Schema Reset. Starting Bulk Upsert...
✅ Database Update Complete: 133 records.


In [9]:
# CELL 5: High-Speed Concurrent NSE Index Fetcher
import requests
import json
import os
import time
import random
import concurrent.futures
from datetime import datetime
from tqdm.notebook import tqdm

# ==========================================
# 1. CONFIGURATION
# ==========================================
OUTPUT_DIR = 'data/indices'
LOG_FILE = 'logs/fast_fetch.log'

# CONCURRENCY SETTINGS
# ⚠️ WARNING: Higher workers = Faster speed, but HIGHER risk of IP Ban.
# Recommended: 3 to 5. Do not go above 10.
MAX_WORKERS = 4 

if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)
if not os.path.exists('logs'): os.makedirs('logs')

# Headers (Mimicking Chrome)
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': '*/*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate',
    'Referer': 'https://www.nseindia.com/market-data/live-equity-market',
    'Connection': 'keep-alive',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-origin',
    'X-Requested-With': 'XMLHttpRequest'
}

URL_HOME = "https://www.nseindia.com"
URL_MASTER_LIST = "https://www.nseindia.com/api/allIndices"
URL_DETAILS_API = "https://www.nseindia.com/api/equity-stockIndices"

# ==========================================
# 2. GLOBAL SESSION MANAGEMENT
# ==========================================
# We use a global session to reuse the TCP connection (Keep-Alive)
session = requests.Session()
session.headers.update(HEADERS)

def init_session():
    """Hits homepage to get cookies"""
    try:
        # random sleep before init to be safe
        time.sleep(1) 
        response = session.get(URL_HOME, timeout=15)
        if response.status_code == 200:
            return True
    except Exception:
        pass
    return False

# ==========================================
# 3. WORKER FUNCTION (The "Engine")
# ==========================================
def fetch_and_save_index(index_name):
    """
    This function runs inside each thread.
    1. Fetches data for one index.
    2. Cleans charts.
    3. Saves to JSON.
    """
    safe_filename = index_name.replace(" ", "_").replace("/", "-")
    file_path = os.path.join(OUTPUT_DIR, f"{safe_filename}.json")
    
    params = {"index": index_name}
    
    try:
        # Random tiny sleep to jitter the threads so they don't hit exact same millisecond
        time.sleep(random.uniform(0.5, 1.5))
        
        response = session.get(URL_DETAILS_API, params=params, timeout=15)
        
        if response.status_code == 200:
            data = response.json()
            
            # --- CLEANING LOGIC ---
            # 1. Remove top-level charts if they exist
            keys_to_remove = ["chartTodayPath", "chart30dPath", "chart365dPath"]
            for k in keys_to_remove:
                data.pop(k, None)
                
            # 2. Remove charts from the specific "data" list items if they exist there
            if "data" in data and isinstance(data["data"], list):
                for stock in data["data"]:
                    if isinstance(stock, dict):
                        for k in keys_to_remove:
                            stock.pop(k, None)

            # --- SAVING ---
            with open(file_path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=4)
                
            return {"status": "success", "index": index_name}
            
        elif response.status_code == 401:
            return {"status": "retry", "index": index_name}
        else:
            return {"status": "error", "index": index_name, "code": response.status_code}
            
    except Exception as e:
        return {"status": "error", "index": index_name, "msg": str(e)}

# ==========================================
# 4. MAIN ORCHESTRATOR
# ==========================================
def main():
    print(f"[{datetime.now()}] 🚀 Starting High-Speed Concurrent Fetcher...")
    
    # --- Phase 1: Get Master List (In-Memory) ---
    print(f"[{datetime.now()}] 🍪 Initializing Session & Fetching Master List...")
    if not init_session():
        print("❌ Failed to initialize session. Check internet/IP.")
        return

    try:
        res = session.get(URL_MASTER_LIST, timeout=15)
        raw_data = res.json()
        
        # Unwrap data if inside "data" key
        if isinstance(raw_data, dict) and "data" in raw_data:
            master_list = raw_data["data"]
        else:
            master_list = raw_data
            
        # Extract just the Index Names (Symbols)
        # logic: prefer 'indexSymbol', fallback to 'index', then 'key'
        index_names = []
        for item in master_list:
            name = item.get("indexSymbol") or item.get("index") or item.get("key")
            if name:
                index_names.append(name)
        
        # Unique names only
        index_names = list(set(index_names))
        print(f"[{datetime.now()}] ✅ Master List Fetched. Found {len(index_names)} indices.")
        
    except Exception as e:
        print(f"❌ Failed to fetch Master List: {e}")
        return

    # --- Phase 2: Concurrent Fetching ---
    print(f"[{datetime.now()}] ⚡ Starting Parallel Fetch (Workers: {MAX_WORKERS})...")
    
    results_log = {"success": 0, "retry": 0, "error": 0}
    
    # ThreadPoolExecutor manages the pool of worker threads
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit all tasks
        future_to_index = {executor.submit(fetch_and_save_index, name): name for name in index_names}
        
        # Process as they complete (using tqdm for progress bar)
        for future in tqdm(concurrent.futures.as_completed(future_to_index), total=len(index_names), desc="Fetching"):
            result = future.result()
            status = result["status"]
            
            if status == "success":
                results_log["success"] += 1
            elif status == "retry":
                # If 401, we might need to re-queue or just log (Simple version: Log it)
                # Implementing re-queue in concurrency is complex, usually just re-run script for missed ones
                results_log["retry"] += 1
            else:
                results_log["error"] += 1

    print("\n" + "="*40)
    print(f"🏁 FINAL REPORT")
    print(f"✅ Success: {results_log['success']}")
    print(f"⚠️ Failed/Auth Error: {results_log['retry']}")
    print(f"❌ Errors: {results_log['error']}")
    print(f"📂 Data saved to: {OUTPUT_DIR}/")
    print("="*40)

if __name__ == "__main__":
    main()

[2025-11-25 03:19:34.643228] 🚀 Starting High-Speed Concurrent Fetcher...
[2025-11-25 03:19:34.647900] 🍪 Initializing Session & Fetching Master List...
[2025-11-25 03:19:36.871285] ✅ Master List Fetched. Found 133 indices.
[2025-11-25 03:19:36.874094] ⚡ Starting Parallel Fetch (Workers: 4)...


Fetching:   0%|          | 0/133 [00:00<?, ?it/s]


🏁 FINAL REPORT
✅ Success: 133
⚠️ Failed/Auth Error: 0
❌ Errors: 0
📂 Data saved to: data/indices/


In [11]:
# CELL 6: The Local Enricher (In-Place Update of Master JSON)
import json
import os
from collections import defaultdict
from tqdm.notebook import tqdm
from datetime import datetime

# ==========================================
# 1. CONFIGURATION
# ==========================================
# We read AND write to the same file to keep it as the Single Source of Truth
MASTER_FILE = 'data/NSE_STOCKS_SKELETON.json'
INDICES_DIR = 'data/indices'

# ==========================================
# 2. BUILD LOOKUP MAP (The "Reverse Index")
# ==========================================
def build_indices_lookup():
    print(f"[{datetime.now()}] 📖 Scanning Index files to build Lookup Map...")
    
    # Map: Symbol -> { "meta": ..., "indices": [list of index names], "web_data": ... }
    lookup_map = defaultdict(lambda: {"meta": None, "indices": []})
    
    if not os.path.exists(INDICES_DIR):
        print(f"❌ Error: {INDICES_DIR} directory not found.")
        return {}

    files = [f for f in os.listdir(INDICES_DIR) if f.endswith('.json')]
    
    for filename in tqdm(files, desc="Scanning Indices"):
        index_name = filename.replace('.json', '').replace('_', ' ')
        file_path = os.path.join(INDICES_DIR, filename)
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = json.load(f)
                
            # Handle Wrapper
            if isinstance(content, dict) and "data" in content:
                stock_list = content["data"]
            elif isinstance(content, list):
                stock_list = content
            else:
                continue

            for stock in stock_list:
                if not isinstance(stock, dict): continue
                
                symbol = stock.get("symbol")
                if not symbol: continue
                
                # Link Stock to Index
                lookup_map[symbol]["indices"].append(index_name)
                
                # Capture the richest metadata
                # We prioritize metadata that has "industry" info
                new_meta = stock.get("meta")
                current_meta = lookup_map[symbol]["meta"]
                
                if new_meta:
                    if current_meta is None:
                        # First time seeing meta, grab it
                        lookup_map[symbol]["web_data"] = stock
                        lookup_map[symbol]["meta"] = new_meta
                    elif new_meta.get("industry") and not current_meta.get("industry"):
                        # Found better meta with industry, upgrade
                        lookup_map[symbol]["web_data"] = stock
                        lookup_map[symbol]["meta"] = new_meta

        except Exception as e:
            print(f"❌ Error reading {filename}: {e}")
            
    return lookup_map

# ==========================================
# 3. UPDATE MASTER JSON IN-PLACE
# ==========================================
def update_master_file(lookup_map):
    if not os.path.exists(MASTER_FILE):
        print(f"❌ Error: {MASTER_FILE} not found. Run Cell 3 first.")
        return

    print(f"[{datetime.now()}] 🔄 Loading Master JSON...")
    with open(MASTER_FILE, 'r', encoding='utf-8') as f:
        master_list = json.load(f)

    print(f"[{datetime.now()}] ⚡ Enriching fields...")
    enriched_count = 0
    
    for stock_obj in tqdm(master_list, desc="Updating Stocks"):
        symbol = stock_obj["master_key"]
        
        # Check if we found this stock in our Indices scan
        if symbol in lookup_map:
            match = lookup_map[symbol]
            web_data = match.get("web_data")
            indices_list = match.get("indices")
            meta = match.get("meta", {})
            
            # --- 1. FILL RAW SOURCE (Source C) ---
            stock_obj["__RAW_DATA_SOURCES__"]["source_nse_web"] = web_data
            
            # --- 2. UPDATE CLASSIFICATION ---
            if meta.get("industry"):
                stock_obj["classification"]["industry"] = meta.get("industry")
            
            # --- 3. UPDATE IDENTIFIERS ---
            if meta.get("slb_isin"):
                stock_obj["identifiers"]["slb_isin"] = meta.get("slb_isin")
            
            if meta.get("activeSeries"):
                stock_obj["identifiers"]["active_series"] = meta.get("activeSeries")

            # --- 4. UPDATE FLAGS ---
            # We use .get(key, False) to ensure it defaults to False if missing
            stock_obj["flags"]["is_fno"] = meta.get("isFNOSec", False)
            stock_obj["flags"]["is_etf"] = meta.get("isETFSec", False)
            stock_obj["flags"]["is_slb_eligible"] = meta.get("isSLBSec", False)
            stock_obj["flags"]["is_suspended"] = meta.get("isSuspended", False)
            stock_obj["flags"]["is_delisted"] = meta.get("isDelisted", False)
            stock_obj["flags"]["is_municipal_bond"] = meta.get("isMunicipalBond", False)
            stock_obj["flags"]["is_mtf_eligible"] = False # Placeholder, Angel API will fill this later
            
            # --- 5. ADD RELATIONS (New Field) ---
            # This helps us query "Get all Nifty 50 stocks" later
            stock_obj["relations"] = {
                "indices": indices_list
            }
            
            enriched_count += 1
        else:
            # If stock not in any index, just add empty relations for consistency
            stock_obj["relations"] = {"indices": []}

    # --- SAVE BACK TO SAME FILE ---
    print(f"[{datetime.now()}] 💾 Saving updates to {MASTER_FILE}...")
    with open(MASTER_FILE, 'w', encoding='utf-8') as f:
        json.dump(master_list, f, indent=4)

    print(f"\n" + "="*50)
    print(f"✅ UPDATE COMPLETE")
    print(f"🔹 File Updated: {MASTER_FILE}")
    print(f"🔹 Stocks Enriched: {enriched_count}/{len(master_list)}")
    print("="*50)

# ==========================================
# 4. EXECUTION
# ==========================================
if __name__ == "__main__":
    lookup_map = build_indices_lookup()
    if lookup_map:
        update_master_file(lookup_map)

[2025-11-25 03:27:11.492963] 📖 Scanning Index files to build Lookup Map...


Scanning Indices:   0%|          | 0/133 [00:00<?, ?it/s]

[2025-11-25 03:27:11.805365] 🔄 Loading Master JSON...
[2025-11-25 03:27:11.849730] ⚡ Enriching fields...


Updating Stocks:   0%|          | 0/2202 [00:00<?, ?it/s]

[2025-11-25 03:27:11.875376] 💾 Saving updates to data/NSE_STOCKS_SKELETON.json...

✅ UPDATE COMPLETE
🔹 File Updated: data/NSE_STOCKS_SKELETON.json
🔹 Stocks Enriched: 889/2202


In [ ]:
# CELL 7: The Token Hunter (Final Fixed Headers & Logic)
import json
import os
import uuid
import requests
import concurrent.futures
import threading
import re
from tqdm.notebook import tqdm
from datetime import datetime

# ==========================================
# 1. CONFIGURATION
# ==========================================
JSON_PATH = 'data/NSE_STOCKS_SKELETON.json'
LOG_DIR = 'logs'
SEARCH_URL = "https://global-search-bbe-prod.angelone.in/v5/search"
MAX_WORKERS = 10 

if not os.path.exists(LOG_DIR): os.makedirs(LOG_DIR)

MISSING_LOG_PATH = os.path.join(LOG_DIR, 'angel_missing_stocks.log')

# ---------------------------------------------------------
# 🚨 TODO: PASTE YOUR FRESH COOKIE HERE FROM BROWSER NETWORK TAB 🚨
# ---------------------------------------------------------
# Instructions:
# 1. Open https://www.angelone.in/ in your browser
# 2. Open Developer Tools (F12)
# 3. Go to Network tab
# 4. Login to Angel One
# 5. Look for any request to angelone.in
# 6. Copy the entire Cookie header value
# 7. Paste it below replacing the placeholder
COOKIE_STRING = "your_cookie_string_here" 

# ==========================================
# 2. EXACT HEADERS FROM YOUR SCREENSHOTS
# ==========================================
HEADERS_TEMPLATE = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/142.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-IN,en;q=0.9,hi;q=0.8",
    "Referer": "https://www.angelone.in/",
    "Origin": "https://www.angelone.in",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-site",
    "App-Version": "sparkweb1",
    "Applicationname": "Spark-Web",
    "X-Sourceid": "3",
    "X-Tokentype": "non_trade_access_token",
    "X-Build-Number": "1",
    # TODO: Copy these from your browser's Network tab (any angelone.in request)
    "X-Deviceid": "your_device_id_here",  # Find in Request Headers
    "X-Clientpublicip": "your_public_ip_here",  # Find in Request Headers
    "Cookie": COOKIE_STRING
}

# Thread-local storage for persistent sessions
thread_local = threading.local()

def get_session():
    if not hasattr(thread_local, "session"):
        thread_local.session = requests.Session()
        thread_local.session.headers.update(HEADERS_TEMPLATE)
    return thread_local.session

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def log_missing(company_name, tried_queries, reason):
    with open(MISSING_LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(f"MISSING: {company_name} | Queries: {tried_queries} | Reason: {reason}\n")

def clean_company_name(name):
    """Removes suffixes like 'Limited', '(India) Ltd'."""
    cleaned = re.sub(r'\s+\(?(Limited|Ltd|India|LTD|LIMITED|INDIA|Plc|Corp|Corporation)\)?\.?$', '', name, flags=re.IGNORECASE)
    cleaned = re.sub(r'\s+\([^)]*\)', '', cleaned)
    return cleaned.strip()

def perform_search(keyword):
    """Hits Angel API."""
    session = get_session()
    # Check if user updated the cookie
    if "YOUR_HUGE_COOKIE" in session.headers['Cookie']:
        print("❌ ERROR: You forgot to paste the fresh Cookie in CONFIGURATION!")
        raise ValueError("Invalid Cookie")

    headers = session.headers.copy()
    headers["X-Requestid"] = str(uuid.uuid4())
    
    # Exact payload matching your browser
    params = {
        "userType": "B2C",
        "accountType": "D",
        "keyword": keyword
    }
    
    try:
        response = session.get(SEARCH_URL, headers=headers, params=params, timeout=5)
        if response.status_code == 200:
            data = response.json()
            if data.get("status") == "success" and "data" in data and "dataList" in data["data"]:
                return data["data"]["dataList"]
    except Exception:
        pass
    return []

def parse_best_match(results, search_symbol=None):
    """Finds the best NSE Equity match."""
    if not results: return None
    
    # 1. Exact Symbol Match
    if search_symbol:
        for item in results:
            if item.get("type") == "scrip" and item.get("symbolName") == search_symbol:
                return item

    # 2. Strict NSE EQ
    for item in results:
        if item.get("type") == "scrip":
            if item.get("scripType") == "nseCash" and item.get("series") == "EQ":
                return item
            # Backup for edge cases
            if item.get("scripType") == "nseCash" and item.get("tradeSymbol", "").endswith("-EQ"):
                return item
    return None

def fetch_and_enrich_stock(stock_obj):
    # Skip if already done
    if stock_obj["trading_info"].get("token_nse") and stock_obj["__RAW_DATA_SOURCES__"].get("source_angel_api"):
        return stock_obj

    identifiers = stock_obj.get("identifiers", {})
    symbol = identifiers.get("symbol")
    company_name = identifiers.get("company_name", "").strip()
    
    if not symbol or not company_name: return stock_obj

    match = None
    tried_queries = []

    # --- 1. Search by Symbol ---
    search_query = symbol
    tried_queries.append(search_query)
    results = perform_search(search_query)
    match = parse_best_match(results, search_symbol=symbol)

    # --- 2. Search by FULL Company Name (Browser Style) ---
    if not match:
        search_query = company_name
        tried_queries.append(search_query)
        results = perform_search(search_query)
        match = parse_best_match(results, search_symbol=symbol)

    # --- 3. Search by Clean Name ---
    if not match:
        cleaned_name = clean_company_name(company_name)
        if cleaned_name != company_name:
            search_query = cleaned_name
            tried_queries.append(search_query)
            results = perform_search(search_query)
            match = parse_best_match(results, search_symbol=symbol)

    # --- UPDATE ---
    if match:
        stock_obj["__RAW_DATA_SOURCES__"]["source_angel_api"] = match
        stock_obj["trading_info"]["token_nse"] = match.get("nseCashToken")
        stock_obj["trading_info"]["token_bse"] = match.get("bseCashToken")
        stock_obj["trading_info"]["tick_size"] = match.get("priceTck")
        if match.get("regLot"):
            try: stock_obj["trading_info"]["lot_size"] = int(match.get("regLot"))
            except: pass
        
        stock_obj["classification"]["sector"] = match.get("sector")
        stock_obj["classification"]["market_cap_category"] = match.get("marketCapCategory")
        stock_obj["classification"]["market_cap_value"] = match.get("mktcap")
        
        if match.get("isMTFEligible") is not None:
            stock_obj["flags"]["is_mtf_eligible"] = match.get("isMTFEligible")
    else:
        log_missing(company_name, tried_queries, "No match found")

    return stock_obj

# ==========================================
# 4. EXECUTION
# ==========================================
def main():
    if not os.path.exists(JSON_PATH):
        print(f"❌ Error: {JSON_PATH} not found.")
        return

    # Clear logs
    open(MISSING_LOG_PATH, 'w').close()

    print(f"[{datetime.now()}] 🔄 Loading Master JSON...")
    with open(JSON_PATH, 'r', encoding='utf-8') as f:
        master_list = json.load(f)

    pending = [s for s in master_list if not s["trading_info"].get("token_nse")]
    print(f"[{datetime.now()}] 🕵️ Sweeping {len(pending)} missing stocks...")
    
    if len(pending) == 0:
        print("✅ No missing stocks to process!")
        return

    updated_list = []
    # Using 5 workers to be gentle with the specific headers
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        updated_list = list(tqdm(executor.map(fetch_and_enrich_stock, master_list), total=len(master_list)))

    success_count = sum(1 for s in updated_list if s["trading_info"].get("token_nse"))

    print(f"[{datetime.now()}] 💾 Saving...")
    with open(JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(updated_list, f, indent=4)

    print(f"\n" + "="*50)
    print(f"✅ COMPLETE")
    print(f"🔹 Total Tokens Found: {success_count}")
    print(f"🔹 Check logs at: {MISSING_LOG_PATH}")
    print("="*50)

if __name__ == "__main__":
    main()

[2025-11-25 03:44:40.851891] 🔄 Loading Master JSON...
[2025-11-25 03:44:40.930196] 🕵️ Sweeping 2202 missing stocks...


  0%|          | 0/2202 [00:00<?, ?it/s]

[2025-11-25 03:45:30.230410] 💾 Saving...

✅ COMPLETE
🔹 Total Tokens Found: 2202
🔹 Check logs at: logs\angel_missing_stocks.log


In [10]:
# CELL 8: Stocks Master Seeder (JSON -> Postgres)
# DEPENDENCY: Requires Cell 1 (Engine) and Cell 4 (Indices Table Populated)

import json
import os
from datetime import datetime
from sqlalchemy import MetaData, Table, Column, String, Integer, Boolean, Float, JSON, ForeignKey, UniqueConstraint, text
from sqlalchemy.dialects.postgresql import insert

# ==========================================
# 1. CONFIGURATION
# ==========================================
JSON_SOURCE = 'data/NSE_STOCKS_SKELETON.json'

# ==========================================
# 2. SCHEMA DEFINITION
# ==========================================
def init_schema(engine):
    metadata = MetaData()
    
    # Reflect existing 'indices' table to allow Foreign Key linking
    try:
        indices_table = Table('indices', metadata, autoload_with=engine)
        print("✅ Detected existing 'indices' table.")
    except Exception as e:
        print(f"❌ CRITICAL ERROR: Table 'indices' not found. Run Cell 4 first!")
        raise e

    # Drop tables to reset schema (Since we are changing columns)
    with engine.begin() as conn:
        conn.execute(text("DROP TABLE IF EXISTS stock_indices_map CASCADE"))
        conn.execute(text("DROP TABLE IF EXISTS stocks_master CASCADE"))

    # --- TABLE: STOCKS_MASTER ---
    stocks_table = Table(
        'stocks_master', metadata,
        Column('id', Integer, primary_key=True, autoincrement=True),
        Column('master_key', String),
        Column('symbol', String, unique=True, nullable=False), # Unique Key
        Column('isin', String),
        Column('company_name', String),
        
        # SERIES FLAGS (Replaces 'active_series' string)
        Column('is_eq', Boolean, default=False), # Equity
        Column('is_be', Boolean, default=False), # Book Entry
        Column('is_bl', Boolean, default=False), # Block Deal
        Column('is_bz', Boolean, default=False), # Trade to Trade
        
        # Trading Info
        Column('token_nse', String),
        Column('token_bse', String),
        Column('tick_size', Float),
        Column('lot_size', Integer),
        Column('listing_date', String),
        Column('currency', String),
        Column('segment', String),
        
        # Classification
        Column('industry', String),
        Column('sector', String),
        Column('market_cap_category', String),
        Column('market_cap_value', Float),
        
        # Other Flags
        Column('is_fno', Boolean),
        Column('is_etf', Boolean),
        Column('is_slb_eligible', Boolean),
        Column('is_mtf_eligible', Boolean),
        Column('is_suspended', Boolean),
        Column('is_delisted', Boolean),
        Column('is_debt', Boolean),
        Column('is_municipal_bond', Boolean),
        
        # Raw Data Vault
        Column('meta', JSON) 
    )
    
    # --- TABLE: STOCK_INDICES_MAP (Relations) ---
    stock_indices_map = Table(
        'stock_indices_map', metadata,
        Column('id', Integer, primary_key=True, autoincrement=True),
        Column('stock_symbol', String, ForeignKey('stocks_master.symbol', ondelete="CASCADE")),
        Column('index_id', Integer, ForeignKey('indices.id', ondelete="CASCADE")),
        UniqueConstraint('stock_symbol', 'index_id', name='uq_stock_index_pair')
    )
    
    metadata.create_all(engine)
    print("✅ Database Schema (stocks_master, stock_indices_map) Created.")
    return stocks_table, stock_indices_map

# ==========================================
# 3. SEEDER LOGIC
# ==========================================
def seed_database():
    if 'engine' not in globals():
        print("❌ Error: Database 'engine' not found. Run Cell 1.")
        return

    if not os.path.exists(JSON_SOURCE):
        print(f"❌ Error: {JSON_SOURCE} not found.")
        return

    print(f"[{datetime.now()}] 📖 Loading Master JSON...")
    with open(JSON_SOURCE, 'r', encoding='utf-8') as f:
        stock_list = json.load(f)

    # Initialize Schema (Fresh Tables)
    stocks_table, relations_table = init_schema(engine)

    # Fetch Index ID Map
    print(f"[{datetime.now()}] 🔗 Fetching Indices Map from DB...")
    index_map = {}
    with engine.connect() as conn:
        result = conn.execute(text("SELECT id, name FROM indices"))
        for row in result:
            index_map[row.name.upper()] = row.id
    
    print(f"   -> Found {len(index_map)} indices available for linking.")

    # Bulk Insert
    print(f"[{datetime.now()}] 🚀 Starting Bulk Upsert for {len(stock_list)} stocks...")
    
    with engine.begin() as conn:
        for item in stock_list:
            # --- Extract Fields ---
            identifiers = item.get("identifiers", {})
            trading = item.get("trading_info", {})
            classif = item.get("classification", {})
            flags = item.get("flags", {})
            
            # Safe conversions
            try: mkt_val = float(classif.get("market_cap_value") or 0)
            except: mkt_val = 0.0
            try: tick = float(trading.get("tick_size") or 0)
            except: tick = 0.0
            
            # --- PARSE SERIES FLAGS ---
            # Ensure it's a list (it might come as a single string or list from different sources)
            raw_series = identifiers.get("active_series", [])
            if isinstance(raw_series, str):
                raw_series = [raw_series]
            
            # Convert to set for O(1) lookup and case-insensitivity
            series_set = set(s.upper() for s in raw_series)
            
            is_eq = "EQ" in series_set
            is_be = "BE" in series_set
            is_bl = "BL" in series_set
            is_bz = "BZ" in series_set

            # --- A. Stock Record ---
            stock_record = {
                "master_key": item.get("master_key"),
                "symbol": item.get("master_key"), 
                "isin": identifiers.get("isin"),
                "company_name": identifiers.get("company_name"),
                
                # New Boolean Columns
                "is_eq": is_eq,
                "is_be": is_be,
                "is_bl": is_bl,
                "is_bz": is_bz,
                
                "token_nse": trading.get("token_nse"),
                "token_bse": trading.get("token_bse"),
                "tick_size": tick,
                "lot_size": trading.get("lot_size"),
                "listing_date": trading.get("listing_date"),
                "currency": trading.get("currency", "INR"),
                "segment": trading.get("segment", "EQUITY"),
                
                "industry": classif.get("industry"),
                "sector": classif.get("sector"),
                "market_cap_category": classif.get("market_cap_category"),
                "market_cap_value": mkt_val,
                
                "is_fno": flags.get("is_fno", False),
                "is_etf": flags.get("is_etf", False),
                "is_slb_eligible": flags.get("is_slb_eligible", False),
                "is_mtf_eligible": flags.get("is_mtf_eligible", False),
                "is_suspended": flags.get("is_suspended", False),
                "is_delisted": flags.get("is_delisted", False),
                "is_debt": flags.get("is_debt", False),
                "is_municipal_bond": flags.get("is_municipal_bond", False),
                
                "meta": item.get("__RAW_DATA_SOURCES__")
            }
            
            # Execute Upsert
            stmt = insert(stocks_table).values(stock_record)
            update_stmt = stmt.on_conflict_do_update(
                index_elements=['symbol'],
                set_={
                    "token_nse": stmt.excluded.token_nse,
                    "token_bse": stmt.excluded.token_bse,
                    "meta": stmt.excluded.meta,
                    "industry": stmt.excluded.industry,
                    "sector": stmt.excluded.sector,
                    "market_cap_value": stmt.excluded.market_cap_value,
                    "is_fno": stmt.excluded.is_fno,
                    # Update new flags
                    "is_eq": stmt.excluded.is_eq,
                    "is_be": stmt.excluded.is_be,
                    "is_bl": stmt.excluded.is_bl,
                    "is_bz": stmt.excluded.is_bz
                }
            )
            conn.execute(update_stmt)
            
            # --- B. Relations ---
            stock_indices = item.get("relations", {}).get("indices", [])
            
            for index_name in stock_indices:
                idx_id = index_map.get(index_name.upper())
                if idx_id:
                    rel_record = {
                        "stock_symbol": item.get("master_key"),
                        "index_id": idx_id
                    }
                    rel_stmt = insert(relations_table).values(rel_record)
                    rel_update = rel_stmt.on_conflict_do_nothing()
                    conn.execute(rel_update)

    print(f"\n" + "="*50)
    print(f"🎉 DATABASE SEEDING COMPLETE!")
    print(f"🔹 Master Table: 'stocks_master' (Updated with Boolean Series)")
    print(f"🔹 Relations Table: 'stock_indices_map'")
    print("="*50)

if __name__ == "__main__":
    seed_database()

[2025-11-25 14:00:08.455938] 📖 Loading Master JSON...
✅ Detected existing 'indices' table.
✅ Database Schema (stocks_master, stock_indices_map) Created.
[2025-11-25 14:00:08.696890] 🔗 Fetching Indices Map from DB...
   -> Found 133 indices available for linking.
[2025-11-25 14:00:08.700749] 🚀 Starting Bulk Upsert for 2202 stocks...

🎉 DATABASE SEEDING COMPLETE!
🔹 Master Table: 'stocks_master' (Updated with Boolean Series)
🔹 Relations Table: 'stock_indices_map'
